# Laboratorio #9

* Josue Say - 228801
* Flavio Galán - 22386

## Repositorio

- [Enlace](https://github.com/JosueSay/labs-ds/tree/main/lab9)
- [Data](https://www.ine.gob.gt/bases-de-datos/accidentes-de-transito/)

## Librerías

In [1]:
import os
import re
import glob
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# PySpark
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, VectorAssembler, StandardScaler, PCA
)
from pyspark.ml.clustering import KMeans
from pyspark.ml.stat import Correlation
from pyspark.ml.classification import (
    RandomForestClassifier, LogisticRegression, DecisionTreeClassifier
)
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator, RegressionEvaluator
)
from pyspark.ml.regression import LinearRegression

# Configuración de gráficos
plt.rcParams["figure.figsize"] = (8,5)
plt.rcParams["axes.grid"] = True

## Constantes

In [2]:
def ensureDir(path):
    """Crea un directorio si no existe y avisa si lo crea o si ya existía."""
    if os.path.exists(path):
        if os.path.isdir(path):
            print(f"El directorio ya existe: {path}")
        else:
            # Existe un archivo en la misma ruta, lo eliminamos
            os.remove(path)
            os.makedirs(path, exist_ok=True)
            print(f"Se eliminó un archivo y se creó el directorio: {path}")
    else:
        os.makedirs(path, exist_ok=True)
        print(f"Directorio creado: {path}")

In [3]:
DATA_BASE = "/opt/app/notebooks/data"

# Data que almacena los csv
HECHOS_BASE = os.path.join(DATA_BASE, "hechos")
VEHICULOS_BASE = os.path.join(DATA_BASE, "vehiculos")
FL_BASE = os.path.join(DATA_BASE, "fallecidos_lesionados")

# Caché
CACHE_BASE = "/opt/app/notebooks/cache"
CACHE_PARQUET = "cache_parquet.txt"
CACHE_COMBINED = "cache_combined.txt"


# Preparación archivos unificados
HECHOS_CSV = os.path.join(HECHOS_BASE, "hechos_combinado.csv")
VEHICULOS_CSV = os.path.join(VEHICULOS_BASE, "vehiculos_combinado.csv")
FL_CSV = os.path.join(FL_BASE, "fl_combinado.csv")

HECHOS_COLS = ["num_corre", "anio_ocu", "dia_ocu", "hora_ocu", "g_hora", "g_hora_5", "mes_ocu", "dia_sem_ocu", "depto_ocu", "mupio_ocu", "zona_ocu", "tipo_veh", "marca_veh", "color_veh", "modelo_veh", "g_modelo_veh", "tipo_eve"]
VEHICULOS_COLS = ["mes_ocu", "dia_sem_ocu", "hora_ocu", "depto_ocu", "mupio_ocu", "zona_ocu", "mayor_menor", "tipo_veh", "color_veh", "modelo_veh", "marca_veh", "g_edad", "anio_ocu_file", "tipo_eve", "estado_con", "g_modelo_veh", "sexo_per", "edad_per", "num_corre", "g_hora", "anio_ocu_dataset", "anio_ocu", "edad_grupo"]
FL_COLS = ["mes_ocu", "dia_sem_ocu", "hora_ocu", "g_hora", "depto_ocu", "mupio_ocu", "zona_ocu", "mayor_menor", "tipo_veh", "color_veh", "modelo_veh", "marca_veh", "anio_ocu_file", "tipo_eve", "edad_grupo", "fall_les", "num_corre", "anio_ocu_dataset", "anio_ocu", "sexo_vic", "edad_vic", "int_o_noint"]

# Aseguramos que existan las carpetas
# ensureDir(DATA_BASE)
# ensureDir(CACHE_BASE)
# ensureDir(RAW_BASE)
# ensureDir(VARIABLE_BASE)
# ensureDir(HECHOS_BASE)
# ensureDir(VEHICULOS_BASE)
# ensureDir(FL_BASE)

In [4]:
def initSpark(appName="labSqlTransformations"):
    return (
        SparkSession.builder
        .master("local[*]")
        .appName(appName)
        .config("spark.sql.session.timeZone", "America/Guatemala")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.sql.files.maxPartitionBytes", 128 * 1024 * 1024)
        .getOrCreate()
    )

In [5]:
def runParquetGeneration():
    """Genera los archivos Parquet desde los CSV limpios, solo si no se ha hecho antes."""
    cacheFilePath = os.path.join(CACHE_BASE, CACHE_PARQUET)
    PARQ_DIR = os.path.join(DATA_BASE, "parquet")

    # Si ya hay caché, no repetir
    if os.path.exists(cacheFilePath):
        print("Archivo de caché encontrado. Ya se ha ejecutado el proceso de generación parquet.")
        return PARQ_DIR

    # Asegurar carpeta Parquet limpia
    if os.path.exists(PARQ_DIR):
        print(f"Eliminando carpeta existente: {PARQ_DIR}")
        shutil.rmtree(PARQ_DIR)
    ensureDir(PARQ_DIR)

    print("Generando archivos Parquet desde CSV limpios...")

    # Leer CSV limpios
    hechosRaw = spark.read.option("header", True).option("inferSchema", True).csv(HECHOS_CSV)
    vehiculosRaw = spark.read.option("header", True).option("inferSchema", True).csv(VEHICULOS_CSV)
    flRaw = spark.read.option("header", True).option("inferSchema", True).csv(FL_CSV)

    # Guardar en Parquet (sobrescribir siempre)
    hechosRaw.write.mode("overwrite").parquet(os.path.join(PARQ_DIR, "hechos"))
    vehiculosRaw.write.mode("overwrite").parquet(os.path.join(PARQ_DIR, "vehiculos"))
    flRaw.write.mode("overwrite").parquet(os.path.join(PARQ_DIR, "fallecidos_lesionados"))

    # Crear caché de control
    with open(cacheFilePath, "w") as f:
        f.write("Parquet generado correctamente.\n")

    print("Parquet generado y caché creado en:", cacheFilePath)
    return PARQ_DIR


In [6]:
def buildCombined(hechos, vehiculos, fl):
    keys = ["anio_ocu","mes_ocu","dia_sem_ocu","hora_ocu","zona_ocu","depto_ocu","tipo_eve"]

    # Asegura tipos básicos
    def _cast_int(df, cols):
        for c in cols:
            if c in df.columns:
                df = df.withColumn(c, F.col(c).cast("int"))
        return df

    hechos    = _cast_int(hechos,    ["anio_ocu","mes_ocu","hora_ocu","depto_ocu"])
    vehiculos = _cast_int(vehiculos, ["anio_ocu","mes_ocu","hora_ocu","depto_ocu"])
    fl        = _cast_int(fl,        ["anio_ocu","mes_ocu","hora_ocu","depto_ocu"])

    # Base hechos (una fila por grupo)
    hechos_k = hechos.select(*[c for c in keys if c in hechos.columns]).dropDuplicates(keys)

    # Conteo de vehículos
    veh_k = (vehiculos
             .groupBy(*[c for c in keys if c in vehiculos.columns])
             .agg(F.count(F.lit(1)).alias("num_vehiculos")))

    # Suma de lesionados/fallecidos (usa tu columna 'fall_les')
    fl_k = (fl
            .groupBy(*[c for c in keys if c in fl.columns])
            .agg(
                F.sum(F.when(F.lower(F.col("fall_les"))=="lesionado", 1).otherwise(0)).alias("lesionados"),
                F.sum(F.when(F.lower(F.col("fall_les"))=="fallecido", 1).otherwise(0)).alias("fallecidos")
            ))

    combined = (hechos_k
                .join(veh_k, keys, "left")
                .join(fl_k,  keys, "left")
                .fillna({"num_vehiculos":0, "lesionados":0, "fallecidos":0}))

    return combined

In [7]:
def runCombinedGeneration(PARQ_DIR, hechos, vehiculos, fl):
    cacheFilePath = os.path.join(CACHE_BASE, CACHE_COMBINED)
    out_path = os.path.join(PARQ_DIR, "combined")

    if os.path.exists(cacheFilePath):
        print("Archivo de caché encontrado. Ya se ha ejecutado el proceso de generación de combinación.")
        return out_path

    # overwrite seguro
    if os.path.exists(out_path):
        print(f"Eliminando combinado previo: {out_path}")
        shutil.rmtree(out_path)

    print("Construyendo DataFrame combinado…")
    combined = buildCombined(hechos, vehiculos, fl)

    combined.write.mode("overwrite").parquet(out_path)
    with open(cacheFilePath, "w") as f:
        f.write("combined parquet ok\n")

    print("Combinado guardado en:", out_path)
    return out_path

In [8]:
spark = initSpark()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/09 22:57:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
PARQ_DIR = runParquetGeneration()

Archivo de caché encontrado. Ya se ha ejecutado el proceso de generación parquet.


In [10]:
hechos = spark.read.parquet(os.path.join(PARQ_DIR, "hechos"))
vehiculos = spark.read.parquet(os.path.join(PARQ_DIR, "vehiculos"))
fl = spark.read.parquet(os.path.join(PARQ_DIR, "fallecidos_lesionados"))

hechos.show(5)

+-------+-----------+--------+------+---------+---------+--------+-----------+---------+----------+----------+-------------+--------+---------+----------------+--------+
|mes_ocu|dia_sem_ocu|hora_ocu|g_hora|depto_ocu|mupio_ocu|zona_ocu|   tipo_veh|color_veh|modelo_veh| marca_veh|anio_ocu_file|tipo_eve|num_corre|anio_ocu_dataset|anio_ocu|
+-------+-----------+--------+------+---------+---------+--------+-----------+---------+----------+----------+-------------+--------+---------+----------------+--------+
|  enero|     martes|       3|     1|        1|      106|       6|    pick up| ignorado|      9999|       jmc|         2013|  choque|        1|            NULL|    2013|
|  enero|     martes|       5|     1|        1|      101|      11|  automovil|   blanco|      2010|    avanti|         2013|  choque|        2|            NULL|    2013|
|  enero|     martes|       3|     1|        1|      108|       3|  automovil|     gris|      9999|   genesis|         2013|colision|        3|       

In [11]:
combined_path = runCombinedGeneration(PARQ_DIR, hechos, vehiculos, fl)
combined = spark.read.parquet(combined_path)

combined.printSchema()
combined.show(10, truncate=False)

Archivo de caché encontrado. Ya se ha ejecutado el proceso de generación de combinación.
root
 |-- anio_ocu: integer (nullable = true)
 |-- mes_ocu: integer (nullable = true)
 |-- dia_sem_ocu: string (nullable = true)
 |-- hora_ocu: integer (nullable = true)
 |-- zona_ocu: integer (nullable = true)
 |-- depto_ocu: integer (nullable = true)
 |-- tipo_eve: string (nullable = true)
 |-- num_vehiculos: long (nullable = true)
 |-- lesionados: long (nullable = true)
 |-- fallecidos: long (nullable = true)

+--------+-------+-----------+--------+--------+---------+--------+-------------+----------+----------+
|anio_ocu|mes_ocu|dia_sem_ocu|hora_ocu|zona_ocu|depto_ocu|tipo_eve|num_vehiculos|lesionados|fallecidos|
+--------+-------+-----------+--------+--------+---------+--------+-------------+----------+----------+
|2013    |NULL   |martes     |0       |99      |19       |choque  |0            |0         |0         |
|2013    |NULL   |martes     |7       |99      |13       |vuelco  |0          